# Problem Book P1, Chapter 11: Walks, Chains and Groups (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools, collections
from math import comb, factorial, log, sqrt, pi, cos
from fractions import Fraction as F
import numpy as np
import matplotlib.pyplot as plt
random.seed(1); rng = np.random.default_rng(1)

def compose(p, q):            # (p*q)(i) = p(q(i)): first q, then p
    return tuple(p[i] for i in q)
def is_even(p):
    seen, s = set(), 0
    for i in range(len(p)):
        if i not in seen:
            j, L = i, 0
            while j not in seen: seen.add(j); j = p[j]; L += 1
            s += L - 1
    return s % 2 == 0
def walk_matrix(G, steps):    # P(g, g*s) = mu(s), group elements as tuples
    idx = {g: i for i, g in enumerate(G)}; P = np.zeros((len(G), len(G)))
    for g in G:
        for s, w in steps.items(): P[idx[g], idx[compose(g, s)]] += w
    return P, idx
def hitting_times(adj, target):   # expected hitting times of target for the simple walk on a graph {v: [nbrs]}
    V = list(adj); i = {v: k for k, v in enumerate(V)}; n = len(V)
    A = np.eye(n); b = np.ones(n)
    for v in V:
        if v == target: b[i[v]] = 0; continue
        for w in adj[v]: A[i[v], i[w]] -= 1 / len(adj[v])
    h = np.linalg.solve(A, b); return {v: h[i[v]] for v in V}
def resistance(adj, a, b):
    V = list(adj); i = {v: k for k, v in enumerate(V)}; n = len(V); L = np.zeros((n, n))
    for v in V:
        for w in adj[v]: L[i[v], i[w]] -= 1; L[i[v], i[v]] += 1
    e = np.zeros(n); e[i[a]], e[i[b]] = 1, -1
    return e @ np.linalg.pinv(L) @ e

Some graphs: the cube, the octahedron, the icosahedron, the dodecahedron and the Petersen graph, as adjacency lists.

In [ ]:
def from_edges(E):
    adj = collections.defaultdict(list)
    for a, b in E: adj[a].append(b); adj[b].append(a)
    return dict(adj)
def hypercube(m): return {v: [v ^ (1 << k) for k in range(m)] for v in range(2**m)}
octa = from_edges([(a, b) for a in range(6) for b in range(a + 1, 6) if (a % 3) != (b % 3)])
# icosahedron: vertices (0, +-1, +-phi) and cyclic permutations; neighbours at distance 2
phi = (1 + sqrt(5)) / 2
P12 = []
for s1 in (1, -1):
    for s2 in (1, -1):
        for v in ((0, s1, s2 * phi), (s1, s2 * phi, 0), (s2 * phi, 0, s1)): P12.append(np.array(v))
icosa = from_edges([(a, b) for a in range(12) for b in range(a + 1, 12) if abs(np.linalg.norm(P12[a] - P12[b]) - 2) < 1e-9])
# dodecahedron: the faces of the icosahedron are its vertices, adjacent when they share an edge
faces = [f for f in itertools.combinations(range(12), 3) if all(b in icosa[a] for a, b in itertools.combinations(f, 2))]
dodeca = from_edges([(i, j) for i in range(20) for j in range(i + 1, 20) if len(set(faces[i]) & set(faces[j])) == 2])
petersen = from_edges([(a, b) for a, b in itertools.combinations(list(itertools.combinations(range(5), 2)), 2) if not set(a) & set(b)])
for name, G in [("cube", hypercube(3)), ("octa", octa), ("icosa", icosa), ("dodeca", dodeca), ("petersen", petersen)]:
    print(name, len(G), sum(len(v) for v in G.values()) // 2, sorted(set(len(v) for v in G.values())))

## Part A: checks

**Problem 11.1 (return probabilities).** A symmetric walk on $\mathfrak S_4$: $p_{2t}(e)$ is the maximum of $p_{2t}$, at least $\frac1{24}$, non-increasing; and the identity
$\sum_g(p_t(g)-\frac1{|G|})^2=p_{2t}(e)-\frac1{|G|}$ of the leads.

In [ ]:
S4 = list(itertools.permutations(range(4)))
steps = {(1, 0, 2, 3): 0.3, (0, 2, 1, 3): 0.3, (1, 2, 3, 0): 0.2, (3, 0, 1, 2): 0.2}   # symmetric: a 4-cycle and its inverse
P, idx = walk_matrix(S4, steps); e = idx[(0, 1, 2, 3)]
p = np.zeros(24); p[e] = 1; rows = []
for t in range(1, 13):
    p = p @ P
    if t % 2 == 0: rows.append(p.copy())
print([round(r[e], 5) for r in rows])
print(all(r.max() == r[e] for r in rows), all(r[e] >= 1 / 24 for r in rows))
pt = np.zeros(24); pt[e] = 1
for t in range(3): pt = pt @ P
print(round(((pt - 1 / 24)**2).sum(), 10), round(rows[2][e] - 1 / 24, 10))

**Problem 11.2 (periodicity).** The walk $x\mapsto2x$ or $3x$ modulo $p$: period computed from the chain against $[G:\langle3\cdot2^{-1}\rangle]$; primes below $200$ where it fails to
equidistribute.

In [ ]:
def order(a, p):
    k, x = 1, a % p
    while x != 1: x = x * a % p; k += 1
    return k
def chain_period(p):
    reach = {1: {0}}; frontier = {1}; times = collections.defaultdict(set); times[1].add(0)
    # period = gcd of t such that the walk can be at 1 at time t, t <= 4p
    cur = {1}; ts = []
    for t in range(1, 4 * p):
        cur = {x * g % p for x in cur for g in (2, 3)}
        if 1 in cur: ts.append(t)
    return math.gcd(*ts)
bad = []
for p in [q for q in range(5, 200) if all(q % r for r in range(2, int(q**0.5) + 1))]:
    gen = len({pow(2, a, p) * pow(3, b, p) % p for a in range(p) for b in range(p)}) == p - 1
    if not gen: continue
    q = 3 * pow(2, -1, p) % p
    d = (p - 1) // order(q, p)
    assert d == chain_period(p)
    if d > 1: bad.append((p, d))
print(bad)

**Problem 11.3 (rolling die).** Orientations reachable at the starting square (a breadth-first search on a $13\times13$ board), the two loops of (c), and
the expected return time $24$ of the orientation.

In [ ]:
RE = np.array([[0, 0, 1], [0, 1, 0], [-1, 0, 0]]); RN = np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]])
moves = {(1, 0): RE, (-1, 0): RE.T, (0, 1): RN, (0, -1): RN.T}
key = lambda M: tuple(M.flatten())
start = (0, 0, key(np.eye(3, dtype=int))); seen = {start}; stack = [start]
while stack:
    x, y, k = stack.pop(); M = np.array(k).reshape(3, 3)
    for (dx, dy), Q in moves.items():
        s = (x + dx, y + dy, key(Q @ M))
        if max(abs(s[0]), abs(s[1])) <= 6 and s not in seen: seen.add(s); stack.append(s)
print(len({k for _, _, k in seen}), len({k for x, y, k in seen if x == y == 0}))
L1 = moves[(0, -1)] @ moves[(-1, 0)] @ moves[(0, 1)] @ moves[(1, 0)]
L2 = moves[(0, 1)] @ moves[(-1, 0)] @ moves[(0, -1)] @ moves[(1, 0)]
print(L1, np.trace(L1)); print(L2, np.trace(L2))
O = sorted({k for _, _, k in seen}); io = {k: i for i, k in enumerate(O)}
Pm = np.zeros((24, 24))
for k in O:
    for Q in moves.values(): Pm[io[k], io[key(Q @ np.array(k).reshape(3, 3))]] += 0.25
w, v = np.linalg.eig(Pm.T); pi_ = np.real(v[:, np.argmin(abs(w - 1))]); pi_ /= pi_.sum()
print("stationary law uniform:", np.allclose(pi_, 1 / 24), " return time:", round(1 / pi_[io[start[2]]], 6))

**Problem 11.4 (icosahedron).** The walk on $A_5$ by $3$-cycles: return probabilities against $\frac1{60}(1+16\cdot4^{-t}+25(-\frac15)^t)$, and the probability of a fifth of a turn
against $\frac15(1-4^{1-t})$.

In [ ]:
A5 = [p for p in itertools.permutations(range(5)) if is_even(p)]
def order_perm(p):
    k, q = 1, p
    while q != tuple(range(5)): q = compose(p, q); k += 1
    return k
C3 = [g for g in A5 if order_perm(g) == 3]
P, idx = walk_matrix(A5, {g: 1 / len(C3) for g in C3})
e = idx[tuple(range(5))]; g5 = (1, 2, 3, 4, 0)
cls = [g for g in A5 if any(compose(compose(h, g5), tuple(np.argsort(h))) == g for h in A5)]   # conjugacy class of g5
print(len(A5), len(C3), len(cls))
p = np.zeros(60); p[e] = 1
for t in range(1, 7):
    p = p @ P
    print(t, round(p[e], 8), round((1 + 16 * 4.0**-t + 25 * (-0.2)**t) / 60, 8), round(sum(p[idx[g]] for g in cls), 8), round((1 - 4.0**(1 - t)) / 5, 8))

**Problem 11.5 (gap and diameter).** The true gap against $\frac1{|S|d^2}$ for $\Z/n\Z$ and for adjacent transpositions, and the formula $\frac2{n-1}(1-\cos\frac\pi n)$.

In [ ]:
for n in (6, 10, 20):
    print("Z/%d" % n, round(1 - cos(2 * pi / n), 5), ">=", round(1 / (2 * (n // 2)**2), 5))
for n in (3, 4, 5, 6):
    G = list(itertools.permutations(range(n)))
    st = {tuple(list(range(i)) + [i + 1, i] + list(range(i + 2, n))): 1 / (n - 1) for i in range(n - 1)}
    P, _ = walk_matrix(G, st); ev = sorted(np.linalg.eigvalsh(P), reverse=True)
    print("S_%d" % n, round(1 - ev[1], 6), round(2 / (n - 1) * (1 - cos(pi / n)), 6), ">=", round(1 / ((n - 1) * comb(n, 2)**2), 6))

**Problem 11.6 (free group).** Exact $P(D_{2n}=0)$ from the distance chain: the return probability $\frac13$, the bounds of (c) and (d), and the $2n$-th roots.

In [ ]:
N = 400; d = np.zeros(N + 3); d[0] = 1; ret = {}
for t in range(1, N + 1):
    nd = np.zeros_like(d); nd[1] += d[0]; nd[2:] += 0.75 * d[1:-1]; nd[:-1] += 0.25 * d[1:]; d = nd
    if t % 2 == 0: ret[t // 2] = d[0]
# first-return probabilities by the renewal equation
u = [1.0] + [ret.get(k, 0) for k in range(1, N // 2 + 1)]; f = [0.0]
for k in range(1, len(u)): f.append(u[k] - sum(f[j] * u[k - j] for j in range(1, k)))
print("P(return) ~", round(sum(f), 6))
for n in (5, 20, 100, 200):
    print(n, ret[n] <= (sqrt(3) / 2)**(2 * n), ret[n] >= (0.75)**(n - 1) / (8 * n * n), round(ret[n]**(1 / (2 * n)), 4), round(sqrt(3) / 2, 4))

**Problem 11.7 (commute times).** $\mathbb E_a[T_b]+\mathbb E_b[T_a]=2|\mathcal E|R(a,b)$ for random pairs in random graphs, and the cover time bound for the cycle and the
Petersen graph (simulated).

In [ ]:
for trial in range(3):
    n = 9; E = [(a, b) for a in range(n) for b in range(a + 1, n) if rng.random() < 0.4] + [(k, k + 1) for k in range(n - 1)]
    G = from_edges(set(E)); m = sum(len(v) for v in G.values()) // 2
    a, b = rng.choice(n, 2, replace=False)
    print(round(hitting_times(G, b)[a] + hitting_times(G, a)[b], 6), round(2 * m * resistance(G, a, b), 6))
def cover(G, start, R=4000):
    tot = 0
    for _ in range(R):
        x, seen, t = start, {start}, 0
        while len(seen) < len(G): x = random.choice(G[x]); seen.add(x); t += 1
        tot += t
    return tot / R
print("Petersen cover", round(cover(petersen, (0, 1)), 2), "<=", 2 * 15 * 9)

**Problem 11.8 (Foster).** Sums of edge resistances, and the hitting times of a neighbour in the symmetric graphs.

In [ ]:
for name, G in [("cube", hypercube(3)), ("petersen", petersen), ("dodeca", dodeca), ("icosa", icosa), ("Q5", hypercube(5))]:
    n = len(G); E = [(a, b) for a in G for b in G[a] if str(a) < str(b)]
    a = next(iter(G)); b = G[a][0]
    print(name, round(sum(resistance(G, x, y) for x, y in E), 6), n - 1, round(hitting_times(G, b)[a], 6))

**Problem 11.9 (opposite vertex).** Hitting times of the farthest vertex and the resistances $\frac56,\frac12,\frac76$; the formula for the $m$-cube against $2^m$.

In [ ]:
def far(G, a):
    dist = {a: 0}; q = [a]
    for v in q:
        for w in G[v]:
            if w not in dist: dist[w] = dist[v] + 1; q.append(w)
    return max(dist, key=dist.get)
for name, G in [("octa", octa), ("cube", hypercube(3)), ("icosa", icosa), ("dodeca", dodeca)]:
    a = next(iter(G)); b = far(G, a); m = sum(len(v) for v in G.values()) // 2
    print(name, round(hitting_times(G, b)[a], 6), F(round(resistance(G, a, b) * 60), 60))
for m in range(1, 11):
    f = sum(F(sum(comb(m, j) for j in range(k)), comb(m - 1, k - 1)) for k in range(1, m + 1))
    h = hitting_times(hypercube(m), 2**m - 1)[0] if m <= 8 else None
    print(m, f, None if h is None else round(h, 6), round(float(f) / 2**m, 4))

**Problem 11.10 (Rayleigh).** $R(0,z_n)$ for the grid with merged boundary against $\frac18\ln(2n+1)$, and for a random subgraph (each edge kept with probability $0.7$).

In [ ]:
def grid_resistance(n, keep=1.0):
    pts = [(x, y) for x in range(-n, n + 1) for y in range(-n, n + 1)]
    def node(p): return "z" if max(abs(p[0]), abs(p[1])) == n else p
    G = collections.defaultdict(list)
    for (x, y) in pts:
        for q in ((x + 1, y), (x, y + 1)):
            if max(abs(q[0]), abs(q[1])) <= n and rng.random() < keep:
                a, b = node((x, y)), node(q)
                if a != b: G[a].append(b); G[b].append(a)
    # keep the component of 0
    comp = {(0, 0)}; st = [(0, 0)]
    while st:
        v = st.pop()
        for w in G[v]:
            if w not in comp: comp.add(w); st.append(w)
    if "z" not in comp: return float("inf")
    H = {v: [w for w in G[v] if w in comp] for v in comp}
    return resistance(H, (0, 0), "z")
for n in (2, 4, 8, 12):
    print(n, round(grid_resistance(n), 4), round(log(2 * n + 1) / 8, 4), round(grid_resistance(n, 0.7), 4))

**Problem 11.11 (Aldous--Broder).** First-entrance trees of a small graph, against the number of spanning trees (matrix-tree theorem).

In [ ]:
def ab_tree(G, r):
    x, seen, T = r, {r}, []
    while len(seen) < len(G):
        y = random.choice(G[x])
        if y not in seen: seen.add(y); T.append(frozenset((x, y)))
        x = y
    return frozenset(T)
G = from_edges([(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 2), (1, 3)])
L = np.diag([len(G[v]) for v in range(5)]).astype(float)
for v in G:
    for w in G[v]: L[v, w] -= 1
tau = round(np.linalg.det(L[1:, 1:]))
for r in (0, 3):
    c = collections.Counter(ab_tree(G, r) for _ in range(60000))
    print("start", r, "trees seen:", len(c), "tau:", tau, "min/max frequency x tau:", round(min(c.values()) * tau / 60000, 3), round(max(c.values()) * tau / 60000, 3))

**Problem 11.12 (Terras).** Parity vectors modulo $2^k$ are all distinct, and the answers $31$ and $33$ of (d).

In [ ]:
def T(m): return m // 2 if m % 2 == 0 else (3 * m + 1) // 2
def pv(m, k):
    v = []
    for _ in range(k): v.append(m % 2); m = T(m)
    return tuple(v)
print(all(len({pv(r, k) for r in range(2**k)}) == 2**k for k in range(1, 15)))
print(next(m for m in range(1, 100) if pv(m, 5) == (1,) * 5), [m for m in range(1, 100) if pv(m, 5) == pv(1, 5)][:3])

**Problem 11.13 (stopping times).** The density of $\{\sigma\le k\}$ among $m<2\cdot10^5$ against the binomial lower bound, and the predicted $6.95\ln m$ steps to reach $1$.

In [ ]:
M = 2 * 10**5; sig = np.zeros(M, int)
for m in range(2, M):
    x, k = m, 0
    while True:
        x = T(x); k += 1
        if x < m: break
    sig[m] = k
for k in (3, 5, 10, 20, 40):
    lb = sum(comb(k, j) for j in range(k + 1) if 3**j < 2**k) / 2**k
    print(k, round(np.mean(sig[2:] <= k), 4), ">=", round(lb, 4))
def steps(m):
    s = 0
    while m != 1: m = T(m); s += 1
    return s
for m0 in (10**4, 10**6):
    print(m0, np.mean([steps(m) for m in range(m0, m0 + 2000)]), round(2 * log(m0) / log(4 / 3), 2))

**Problem 11.14 (carries).** The carries matrix for $n$ numbers in base $b$: $P_b^m=P_{b^m}$, the Eulerian stationary law, and the eigenvalues $b^{-j}$.

In [ ]:
def carries(n, b):
    P = np.zeros((n, n))
    # distribution of the sum of n digits by convolution
    s = np.zeros(1); s[0] = 1
    for _ in range(n): s = np.convolve(s, np.ones(b) / b)
    for i in range(n):
        for tot, w in enumerate(s): P[i, (i + tot) // b] += w
    return P
def eulerian(n): return [sum((-1)**r * comb(n + 1, r) * (k + 1 - r)**n for r in range(k + 2)) for k in range(n)]
for n, b in [(3, 10), (4, 3), (5, 2)]:
    P = carries(n, b)
    print(n, b, np.allclose(np.linalg.matrix_power(P, 3), carries(n, b**3)),
          np.round(np.sort(np.linalg.eigvals(P).real)[::-1], 6), np.allclose(np.array(eulerian(n)) / factorial(n) @ P, np.array(eulerian(n)) / factorial(n)))

**Problem 11.15 (visible points).** $P(\gcd(X_t,Y_t)=1)$ computed exactly with the rotated coordinates, against $\frac8{\pi^2}$ and $\frac4{\pi^2}$; and the bound of (c).

In [ ]:
def visible(t):
    k = np.arange(t + 1); w = np.exp([math.lgamma(t + 1) - math.lgamma(j + 1) - math.lgamma(t - j + 1) - t * log(2) for j in k]); u = 2 * k - t
    X = (u[:, None] + u[None, :]) // 2; Y = (u[:, None] - u[None, :]) // 2
    return (np.outer(w, w)[np.gcd(X, Y) == 1]).sum()
for t in (300, 301, 1500, 1501, 3000, 3001):
    print(t, round(visible(t), 5))
print(round(8 / pi**2, 5), round(4 / pi**2, 5))
for p, t in [(3, 50), (7, 200), (101, 400), (1009, 1000)]:
    exact = sum(cos(2 * pi * k / p)**t for k in range(p)) / p
    print(p, t, round(exact, 6), "<=", round(1 / p + 2 / sqrt(t), 6))

**Problem 11.16 (range).** $\mathbb E[R_t]$ by exhaustive enumeration for small $t$, the formula $(4m+1)u_{2m}$, and $\sqrt{8t/\pi}$.

In [ ]:
u = lambda m: comb(2 * m, m) / 4**m
for t in (2, 4, 6, 10, 14):
    tot = 0
    for s in itertools.product((-1, 1), repeat=t):
        tot += len(set(itertools.accumulate(s, initial=0)))
    print(t, tot / 2**t, (2 * t + 1) * u(t // 2))
for t in (100, 10**4, 10**6):
    print(t, round((2 * t + 1) * u(t // 2), 3) if t <= 10**4 else "", round(sqrt(8 * t / pi), 3))

**Problem 11.17 (cycle cover).** Simulated cover time and last vertex on the $9$-cycle; the star counterexample.

In [ ]:
n = 9; T_ = 0; last = collections.Counter(); R = 40000
for _ in range(R):
    x, seen, t = 0, {0}, 0
    while len(seen) < n:
        x = (x + random.choice((-1, 1))) % n; t += 1
        if x not in seen: seen.add(x); lv = x
    T_ += t; last[lv] += 1
print(T_ / R, n * (n - 1) / 2, [round(last[v] / R, 3) for v in range(1, n)], round(1 / (n - 1), 3))
star = {0: [1, 2, 3], 1: [0], 2: [0], 3: [0]}
c = collections.Counter()
for _ in range(10000):
    x, seen = 1, {1}
    while len(seen) < 4:
        x = random.choice(star[x])
        if x not in seen: seen.add(x); lv = x
    c[lv] += 1
print(dict(c))

**Problem 11.18 (three gamblers).** Exact expected durations by solving the linear system, against $\frac{3abc}s$ and $ab+bc+ca$.

In [ ]:
def gamblers(s):
    states = [(a, b, s - a - b) for a in range(s + 1) for b in range(s + 1 - a)]; idx = {x: i for i, x in enumerate(states)}; n = len(states)
    A1, A2 = np.eye(n), np.eye(n); r1, r2 = np.zeros(n), np.zeros(n)
    for x in states:
        i = idx[x]; alive = [k for k in range(3) if x[k] > 0]; pairs = list(itertools.combinations(alive, 2))
        for A, r, cond in ((A1, r1, len(alive) == 3), (A2, r2, len(alive) >= 2)):
            if not cond: continue
            r[i] = 1
            for (p_, q_) in pairs:
                for d in (1, -1):
                    y = list(x); y[p_] += d; y[q_] -= d; A[i, idx[tuple(y)]] -= 1 / (2 * len(pairs))
    return np.linalg.solve(A1, r1), np.linalg.solve(A2, r2), idx
t1, t2, idx = gamblers(12)
for a, b, c in [(4, 4, 4), (1, 2, 9), (3, 4, 5)]:
    i = idx[(a, b, c)]; print((a, b, c), round(t1[i], 6), 3 * a * b * c / (a + b + c), round(t2[i], 6), a * b + b * c + c * a)

**Problem 11.19 (Moran).** Exact absorption times against the formula, and $t_i/N^2$ against the entropy for $p=0.3$.

In [ ]:
def moran(N):
    A = np.eye(N + 1); r = np.zeros(N + 1)
    for x in range(1, N):
        q = x * (N - x) / N**2; A[x, x] = 2 * q; A[x, x + 1] = A[x, x - 1] = -q; r[x] = 1
    return np.linalg.solve(A, r)
N = 30; t = moran(N)
formula = lambda i: N * i * sum(1 / x for x in range(i, N)) + N * (N - i) * sum(1 / y for y in range(N - i + 1, N))
print([(i, round(t[i], 6), round(formula(i), 6)) for i in (1, 7, 15, 29)])
for N in (100, 400, 1600):
    t = moran(N); p = 0.3
    print(N, round(t[int(p * N)] / N**2, 5), round(-(p * log(p) + (1 - p) * log(1 - p)), 5), round(t[1] / (N * log(N)), 4))

**Problem 11.20 (Bernoulli's urns).** Eigenvalues against $1-\frac{k(2n-k+1)}{n^2}$, the hypergeometric law, and the mean at $t=\frac n4\ln n$.

In [ ]:
for n in (6, 20):
    P = np.zeros((n + 1, n + 1))
    for x in range(n + 1):
        if x: P[x, x - 1] = (x / n)**2
        if x < n: P[x, x + 1] = ((n - x) / n)**2
        P[x, x] = 1 - P[x].sum()
    ev = np.sort(np.linalg.eigvals(P).real)[::-1]; lam = sorted([1 - k * (2 * n - k + 1) / n**2 for k in range(n + 1)], reverse=True)
    pi_ = np.array([comb(n, x)**2 for x in range(n + 1)]) / comb(2 * n, n)
    t = int(n / 4 * log(n)); mean = (np.eye(n + 1)[n] @ np.linalg.matrix_power(P, t)) @ np.arange(n + 1)
    sd = sqrt(pi_ @ (np.arange(n + 1) - n / 2)**2)
    print(n, np.allclose(ev, lam), np.allclose(pi_ @ P, pi_), t, round(mean - n / 2, 4), ">=", round(sqrt(n) / 3, 4), " sd under pi:", round(sd, 4))

**Problem 11.21 (Kac's ring).** Mean and variance of $\Delta_t$ over random rings, against $(1-2\mu)^t\Delta_0$ and $n(2t-1)$; the think box.

In [ ]:
def kac(eps, c, t):
    n = len(eps); col = c.astype(float).copy()
    for _ in range(t): col = np.roll(col * eps, 1)
    return col
n, mu = 300, 0.1
for t in (1, 5, 20, 60):
    vals = [kac(np.where(rng.random(n) < mu, -1, 1), np.ones(n), t).sum() for _ in range(3000)]
    print(t, round(np.mean(vals), 2), round(n * (1 - 2 * mu)**t, 2), round(np.var(vals), 1), "<=", n * (2 * t - 1))
eps = np.array([-1, 1, -1, 1, -1, 1])
print(kac(eps, np.ones(6), 6), kac(eps, np.ones(6), 12), round((1 + 0.8**10) / 2, 4))

**Problem 11.22 (Gilbreath).** The Gilbreath arrangements are the interval-like ones and the ones with distinct residues; there are $2^{n-1}$, each produced
by exactly two pairs ($k$, interleaving).

In [ ]:
def gilbreath(n):
    res = collections.Counter(); deck = list(range(1, n + 1))
    for k in range(n + 1):
        A, B = deck[:k][::-1], deck[k:]
        for mask in itertools.combinations(range(n), k):
            out, ia, ib, ms = [], 0, 0, set(mask)
            for pos in range(n):
                if pos in ms: out.append(A[ia]); ia += 1
                else: out.append(B[ib]); ib += 1
            res[tuple(out)] += 1
    return res
for n in range(2, 9):
    g = gilbreath(n); perms = list(itertools.permutations(range(1, n + 1)))
    resid = {p for p in perms if all(len({x % j for x in p[:j]}) == j for j in range(1, n + 1))}
    interval = {p for p in perms if all(max(p[:j]) - min(p[:j]) == j - 1 for j in range(1, n + 1))}
    print(n, len(g), 2**(n - 1), set(g) == resid == interval, set(g.values()))

## Part B: laboratory

**L11.1 (Loschmidt's reversal).** Take a Kac ring with $n=2000$, $\mu=\frac1{10}$, all balls black, and run it for $t=100$ ticks. Now reverse: run the
backward dynamics for $100$ ticks and watch $\Delta$ climb back to $n$. Then perturb the state at time $100$ by recolouring $1\%$ of the balls at
random before reversing. What happens? Would you expect the same for a real gas?

**L11.2 (uniform spanning trees of a grid).** Sample $60000$ trees of the $3\times3$ grid with Aldous--Broder, from a corner and from the centre,
and compare the frequencies with $\frac1{\tau}$, where $\tau$ comes from the matrix-tree theorem. Then estimate the probability that a given edge
belongs to the tree and compare it with its resistance (the leads of Problem 11.8).

**L11.3 (Collatz in the large).** For $m$ near $10^{k}$, $k=4,\dots,9$, compute the number of steps to reach $1$ for $1000$ random $m$, divide by $\ln m$, and
plot the averages against $\frac2{\ln(4/3)}=6.95$. Plot the histogram of the normalised fluctuations for $k=9$: does it look Gaussian? Explain with
the random model.

**L11.4 (the rolling die comes home).** For the die of Problem 11.3 making random moves, compute exactly (by dynamic programming on a board
large enough) the probability $a_t$ of being back at the start square, and $b_t$ of being back at the start square in the initial
orientation, for even $t\le60$. What does $\frac{b_t}{a_t}$ tend to? Explain the limit.

## Solutions

### L11.1

In [ ]:
n, mu = 2000, 0.1
eps = np.where(np.random.default_rng(7).random(n) < mu, -1, 1)
col = np.ones(n); hist = [col.sum()]
for _ in range(100): col = np.roll(col * eps, 1); hist.append(col.sum())
def back(col):   # undo one tick: move from j+1 to j, change colour if edge j is marked
    return np.roll(col, -1) * eps
c1 = col.copy(); h1 = []
for _ in range(100): c1 = back(c1); h1.append(c1.sum())
c2 = col.copy(); flip = rng.choice(n, n // 100, replace=False); c2[flip] *= -1; h2 = []
for _ in range(100): c2 = back(c2); h2.append(c2.sum())
plt.figure(figsize=(6, 3)); plt.plot(range(101), hist, label="forward"); plt.plot(range(101, 201), h1, label="reversed")
plt.plot(range(101, 201), h2, label="reversed after 1% noise"); plt.legend(); plt.show()
print(hist[-1], h1[-1], h2[-1])

The exact reversal brings every ball back to black, as Loschmidt said it must. Recolouring $20$ balls before reversing only spoils those $20$
balls: the reversed run ends at $\Delta=n-40$. The Kac ring is forgiving because its balls do not interact: an error in one ball never spreads. In
a real gas the molecules collide, and a tiny error in one velocity is amplified exponentially at each collision, so after a short time
the reversed motion has nothing to do with the past. That sensitivity, not the reversibility of the equations, is why nobody ever sees
a gas unmix: the state that leads back to order exists, but it cannot be prepared.

### L11.2

In [ ]:
grid = {(x, y): [(x + dx, y + dy) for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)) if 0 <= x + dx < 3 and 0 <= y + dy < 3] for x in range(3) for y in range(3)}
V = list(grid); iv = {v: k for k, v in enumerate(V)}; Lg = np.zeros((9, 9))
for v in V:
    for w in grid[v]: Lg[iv[v], iv[w]] -= 1; Lg[iv[v], iv[v]] += 1
tau = round(np.linalg.det(Lg[1:, 1:])); print("tau =", tau)
for r in ((0, 0), (1, 1)):
    c = collections.Counter(ab_tree(grid, r) for _ in range(60000))
    fr = np.array(list(c.values())) / 60000
    chi2 = 60000 * ((fr - 1 / tau)**2 * tau).sum() + 60000 * (tau - len(c)) / tau
    print(r, len(c), round(fr.min() * tau, 3), round(fr.max() * tau, 3), "chi2 ~", round(chi2, 1), "(dof", tau - 1, ")")
e1 = frozenset(((0, 0), (1, 0))); e2 = frozenset(((1, 0), (1, 1)))
for e in (e1, e2):
    a, b = tuple(e); print(sorted(e), round(sum(w for T, w in c.items() if e in T) / 60000, 4), round(resistance(grid, a, b), 4))

The $3\times3$ grid has $192$ spanning trees, and both starting points produce each of them with frequency close to $\frac1{192}$; the
chi-square statistic is of the order of its number of degrees of freedom, as it should be for a uniform sample. The probability that an
edge is in the tree matches its effective resistance: about $0.708$ for an edge at a corner, which a tree has few ways to avoid, and $\frac7{12}=0.583$
for an edge at the centre.

### L11.3

In [ ]:
res = []
for k in range(4, 10):
    ms = rng.integers(10**k, 2 * 10**k, 1000)
    st = np.array([steps(int(m)) for m in ms]); res.append((k, (st / np.log(ms)).mean(), st, ms))
print([(k, round(a, 3)) for k, a, _, _ in res], round(2 / log(4 / 3), 3))
k, a, st, ms = res[-1]
z = (st - 2 * np.log(ms) / log(4 / 3)) / np.sqrt(np.log(ms))
plt.figure(figsize=(6, 3)); plt.hist(z, bins=40, density=True); plt.show()
print("mean, sd of normalised fluctuation:", round(z.mean(), 3), round(z.std(), 3))

The averages of $\frac{\text{steps}}{\ln m}$ approach $6.95$ slowly from below (the last steps, near $1$, are not random). In the random model the number of
steps $\tau$ to bring $\ln m$ down to $0$ is a first passage time of a random walk with drift $-0.144$ and step variance $\frac14(\ln3)^2$, so by the
central limit theorem $\frac{\tau-6.95\ln m}{\sqrt{\ln m}}$ is approximately Gaussian with standard deviation $\frac{\frac12\ln3}{0.144^{3/2}}\approx10$. The histogram is
indeed bell-shaped, with a spread of that order: the Collatz map looks random to every statistic we know how to compute.

### L11.4

In [ ]:
B = 64; Os = O; io = {k: i for i, k in enumerate(Os)}
step = {d: np.array([io[key(Q @ np.array(k).reshape(3, 3))] for k in Os]) for d, Q in moves.items()}
P0 = np.zeros((2 * B + 1, 2 * B + 1, 24)); P0[B, B, io[start[2]]] = 1
cur = P0; out = []
for t in range(1, 61):
    nxt = np.zeros_like(cur)
    for (dx, dy), perm in step.items():
        shifted = np.roll(np.roll(cur, dx, axis=0), dy, axis=1)
        tmp = np.zeros_like(shifted); tmp[:, :, perm] = shifted
        nxt += 0.25 * tmp
    cur = nxt
    if t % 2 == 0: out.append((t, cur[B, B].sum(), cur[B, B, io[start[2]]]))
for t, a, b in out[::5]: print(t, round(a, 6), round(b, 6), round(b / a, 5))

The ratio $\frac{b_t}{a_t}$ tends to $\frac1{12}$. Given that the die is back on its square (an even time), its orientation lies in $A_4$, by Problem 11.3(b). As $t$
grows the orientation forgets the path: conditionally on the position, it becomes uniform on the $12$ elements of $A_4$. The precise
statement is a local limit theorem for a random walk on the group of motions $\Z^2\rtimes$(rotations); the uniformity reflects the fact
that closed tours already generate all of $A_4$, Problem 11.3(c). On a board of side $65$ the walk does not feel the edges before $t=32$, and
the wrap-around of `np.roll` is harmless for $t\le60$ because the corrections are tiny.